# First find the huc8 value/s for your area of interest/ Not necessary if you already have one from previous code

In [1]:
import geopandas as gpd
import pandas as pd

# Load AOI shapefile
aoi = gpd.read_file("/content/boundary.shp")

# Load full HUC8 shapefile (in EPSG:5070)
huc8 = gpd.read_file("/content/HUC_8_EPSG_5070.shp")

# Ensure CRS match
if aoi.crs != huc8.crs:
    aoi = aoi.to_crs(huc8.crs)

# Spatial join: HUC8s intersecting AOI
intersecting = gpd.sjoin(huc8, aoi, how="inner", predicate="intersects")

# Extract unique HUC8 codes as sorted strings
unique_huc8_codes = sorted(intersecting["HUC8"].astype(str).unique())

# Save to CSV (with header, preserve leading zeros)
pd.DataFrame(unique_huc8_codes, columns=["HUC8"]).to_csv("HUC8.csv", index=False)

print("✅ Saved to HUC8.csv")


✅ Saved to HUC8.csv


# Find the NWM reach_IDs within the domain

#####Install necessary libraries

In [2]:
!pip install s3fs zarr fsspec xarray geopandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.6/202.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 85.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.3.0 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.3.0 which is incompatible.


##### Find the feature IDs

In [3]:
import s3fs
import xarray as xr
import geopandas as gpd
import zarr

# === Input shapefile ===
input_model_boundary_filename = "boundary.shp"  # replace with your shapefile name

# --- Connect to NOAA NWM dataset on AWS ---
s3 = s3fs.S3FileSystem(anon=True, client_kwargs=dict(region_name="us-east-1"))
store = s3fs.S3Map(
    root="s3://noaa-nwm-retrospective-3-0-pds/CONUS/zarr/chrtout.zarr",
    s3=s3,
    check=False
)

# --- Open dataset using legacy API ---
nwm_ds = xr.open_zarr(store)   # no "engine" here
df = nwm_ds[["feature_id", "latitude", "longitude"]].to_dataframe().reset_index()

# --- Convert to GeoDataFrame ---
nwm_gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude))
nwm_gdf.crs = "EPSG:4326"

# --- Read your model boundary shapefile ---
boundary = gpd.read_file(input_model_boundary_filename)

# --- Ensure both layers have the same CRS ---
if boundary.crs != "EPSG:4326":
    boundary = boundary.to_crs("EPSG:4326")

# --- Spatial join: keep only NWM reaches inside boundary ---
reach_ids_gdf = gpd.sjoin(nwm_gdf, boundary, how="inner", predicate="intersects")

# --- Extract only required columns ---
all_reaches = reach_ids_gdf[["feature_id", "latitude", "longitude"]].drop_duplicates()

# --- Save to CSV ---
all_reaches.to_csv("feature_IDs.csv", index=False)

print("✅ File 'feature_IDs.csv' created with all Feature IDs and coordinates in your shapefile domain.")


✅ File 'feature_IDs.csv' created with all Feature IDs and coordinates in your shapefile domain.


In [4]:
!find /content -type f ! -name '*.csv' -delete


### Make an input files directory

In [5]:
!mkdir /content/input_raster_files


### Upload all the input raster files to content section and run the following code to move them to input folder.

In [6]:
import os
import shutil
import glob

# Define target folder
target_folder = '/content/input_raster_files'
os.makedirs(target_folder, exist_ok=True)

# Get all non-CSV files in /content
files_to_move = [f for f in glob.glob('/content/*') if os.path.isfile(f) and not f.endswith('.csv')]

# Move each file
for file_path in files_to_move:
    filename = os.path.basename(file_path)
    if not file_path.startswith(target_folder):  # Avoid moving into itself
        shutil.move(file_path, os.path.join(target_folder, filename))

print("✅ Files moved to:", target_folder)



✅ Files moved to: /content/input_raster_files


## Preprocessing for resolution, raster type, and nodata values

##### Install necessary libraries

In [7]:
!pip install geopandas rasterio fiona shapely pyproj

## This code is for dealing with projection, resolutions (keeps minimum) of input raster maps and also convert to float 32 type.

In [8]:
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from scipy.ndimage import zoom
import os, glob
import numpy as np

################## Input Files #########################################
########################################################################
input_folder  = "input_raster_files/"
output_folder = "resampled_output/"
TARGET_CRS    = "EPSG:4326"
########################################################################

os.makedirs(output_folder, exist_ok=True)

raster_files = glob.glob(os.path.join(input_folder, "*.tif"))
print(f"Found {len(raster_files)} rasters to process.")
if not raster_files:
    raise SystemExit("No rasters found.")

# --- Step 1: Reproject everything to EPSG:4326 and find finest resolution ---
reprojected_data = {}
finest = None

for raster_path in raster_files:
    filename = os.path.basename(raster_path)

    with rasterio.open(raster_path) as src:
        t, w, h = calculate_default_transform(
            src.crs, TARGET_CRS, src.width, src.height, *src.bounds
        )

        nodata = float(src.nodata) if src.nodata is not None else -9999.0
        count  = src.count

        profile = src.meta.copy()
        profile.update({
            "crs"      : TARGET_CRS,
            "transform": t,
            "width"    : w,
            "height"   : h,
            "dtype"    : "float32",
            "nodata"   : nodata,
        })

        # Reproject each band
        data_reproj = np.zeros((count, h, w), dtype=np.float32)
        for band_idx in range(1, count + 1):
            reproject(
                source       =rasterio.band(src, band_idx),
                destination  =data_reproj[band_idx - 1],
                src_transform=src.transform,
                src_crs      =src.crs,
                dst_transform=t,
                dst_crs      =TARGET_CRS,
                resampling   =Resampling.bilinear
            )

        px     = min(abs(t.a), abs(t.e))
        finest = px if finest is None else min(finest, px)

        reprojected_data[filename] = {
            "data"     : data_reproj,
            "profile"  : profile,
            "transform": t,
            "width"    : w,
            "height"   : h,
            "count"    : count,
            "nodata"   : nodata,
        }

    print(f"  🔄 Reprojected: {filename}  (res: {px:.6f} deg)")

print(f"\nFinest resolution across all rasters: {finest:.8f} degrees")

# --- Step 2: Resample all to finest resolution and save as float32 ---
print("\nResampling to finest resolution...\n")

for filename, d in reprojected_data.items():
    output_path = os.path.join(output_folder, filename)

    t          = d["transform"]
    base_w     = d["width"]
    base_h     = d["height"]
    current_px = min(abs(t.a), abs(t.e))

    scale      = current_px / finest
    new_width  = max(1, int(round(base_w * scale)))
    new_height = max(1, int(round(base_h * scale)))

    new_transform = t * t.scale(
        base_w / new_width,
        base_h / new_height
    )

    # Resample using scipy zoom (bilinear = order=1)
    data_resampled = np.stack([
        zoom(d["data"][b], scale, order=1)
        for b in range(d["count"])
    ]).astype(np.float32)

    profile = d["profile"].copy()
    profile.update({
        "width"    : data_resampled.shape[2],
        "height"   : data_resampled.shape[1],
        "transform": new_transform,
        "dtype"    : "float32",
        "nodata"   : d["nodata"],
    })

    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(data_resampled)

    print(f"  ✅ {filename}  →  shape: {data_resampled.shape}  res: {finest:.8f} deg  dtype: float32")

print("\n🎉 Done: All rasters reprojected to EPSG:4326, resampled to finest resolution, saved as float32.")

Found 24 rasters to process.
  🔄 Reprojected: HEC-RAS2D_18181565_2472_cms_depth.tif  (res: 0.000100 deg)
  🔄 Reprojected: HAND_18181565_1229_cms_depth.tif  (res: 0.000099 deg)
  🔄 Reprojected: Nencarta_18181565_1910_cms_depth.tif  (res: 0.000100 deg)
  🔄 Reprojected: HEC-RAS2D_18181565_1530_cms_depth.tif  (res: 0.000100 deg)
  🔄 Reprojected: HAND_18181565_775_cms_depth.tif  (res: 0.000099 deg)
  🔄 Reprojected: Nencarta_18181565_775_cms_depth.tif  (res: 0.000100 deg)
  🔄 Reprojected: Nencarta_18181565_2192_cms_depth.tif  (res: 0.000100 deg)
  🔄 Reprojected: SRH-2D_18181565_2472_cms_depth.tif  (res: 0.000100 deg)
  🔄 Reprojected: SRH-2D_18181565_2192_cms_depth.tif  (res: 0.000100 deg)
  🔄 Reprojected: Nencarta_18181565_1530_cms_depth.tif  (res: 0.000100 deg)
  🔄 Reprojected: HAND_18181565_1910_cms_depth.tif  (res: 0.000099 deg)
  🔄 Reprojected: Nencarta_18181565_2472_cms_depth.tif  (res: 0.000100 deg)
  🔄 Reprojected: HEC-RAS2D_18181565_1910_cms_depth.tif  (res: 0.000100 deg)
  🔄 Reproje

In [9]:
# @title Run this code to convert all the nodata region values to -99999
import os
import rasterio
import numpy as np
import csv

# Directories
input_dir = '/content/resampled_output'
output_dir = '/content/corrected_rasters'
os.makedirs(output_dir, exist_ok=True)
log_path = os.path.join(output_dir, 'nodata_log.csv')

# Fixed NoData value expected by the visualization tool
FIXED_NODATA = -99999

# Common bad values to treat as NoData
COMMON_BAD_VALUES = [-9999, 0, -3.402823e+38]

# Initialize log file
with open(log_path, 'w', newline='') as log_file:
    writer = csv.writer(log_file)
    writer.writerow(['File Name', 'Original NoData', 'Unusual Values < -1e6'])

# Process all .tif files in input directory
for file_name in os.listdir(input_dir):
    if file_name.endswith(".tif"):
        input_path = os.path.join(input_dir, file_name)
        output_path = os.path.join(output_dir, file_name)

        with rasterio.open(input_path) as src:
            data = src.read(1).astype(np.float32)
            profile = src.profile
            original_nodata = src.nodata

            # Step 1: Convert existing -99999 to NaN (cleanup)
            data[data == FIXED_NODATA] = np.nan

            # Step 2: Build mask for invalid values
            mask = np.isin(data, COMMON_BAD_VALUES)
            if original_nodata is not None:
                if np.isnan(original_nodata):
                    mask |= np.isnan(data)
                else:
                    mask |= (data == original_nodata)
            mask |= np.isnan(data)

            # Step 3: Replace all masked values with -99999
            data[mask] = FIXED_NODATA

            # Step 4: Update profile and save without reprojection
            profile.update(nodata=FIXED_NODATA, compress='LZW')

            with rasterio.open(output_path, 'w', **profile) as dst:
                dst.write(data, 1)

            # Step 5: Log NoData info and unusual values
            unusual_values = np.unique(data[data < -1e6])
            with open(log_path, 'a', newline='') as log_file:
                writer = csv.writer(log_file)
                writer.writerow([file_name, original_nodata, "; ".join(map(str, unusual_values))])

print("✅ All rasters cleaned and saved with NoData = -99999.")
print(f"📄 Log saved to: {log_path}")


✅ All rasters cleaned and saved with NoData = -99999.
📄 Log saved to: /content/corrected_rasters/nodata_log.csv


## Covert depth rasters to extent vectors.

In [10]:
import os
import rasterio
from rasterio.features import shapes
import geopandas as gpd
import numpy as np

# Input and output directories
input_folder = '/content/corrected_rasters'  # ✅ Update as needed
output_folder = '/content/extent_files'
os.makedirs(output_folder, exist_ok=True)

# Loop through all raster files
for filename in os.listdir(input_folder):
    if filename.endswith('depth.tif'):  # Adjust pattern as needed
        raster_path = os.path.join(input_folder, filename)

        with rasterio.open(raster_path) as src:
            image = src.read(1)
            raster_crs = src.crs

            # Step 1: Detect or define NoData value
            nodata_val = src.nodata
            if nodata_val is None:
                nodata_val = -99999  # fallback (common placeholder)

            # Step 2: Build mask to ignore NoData, zeros, and NaNs
            mask = (~np.isnan(image)) & (image != 0) & (image != nodata_val)

            # Step 3: Extract shapes while safely converting values
            results = (
                {
                    "properties": {"value": float(v) if not np.isnan(v) else None},
                    "geometry": s
                }
                for s, v in shapes(image, mask=mask, transform=src.transform)
                if not np.isnan(v)
            )

            geoms = list(results)
            if not geoms:
                print(f"⚠️ Skipping {filename}: No valid geometries found.")
                continue

            # Step 4: Create GeoDataFrame
            gdf = gpd.GeoDataFrame.from_features(geoms)
            gdf.set_crs(raster_crs, inplace=True)

            # Step 5: Generate output name and save shapefile
            base_name = os.path.splitext(filename)[0].replace('_depth', '')
            out_name = f"{base_name}_extent.shp"
            out_path = os.path.join(output_folder, out_name)

            gdf.to_file(out_path)
            print(f"✅ Saved vector: {out_name}")

print(f"\n🎉 Conversion complete. Vector files saved to: {output_folder}")


✅ Saved vector: HEC-RAS2D_18181565_2472_cms_extent.shp
✅ Saved vector: HAND_18181565_1229_cms_extent.shp
✅ Saved vector: Nencarta_18181565_1910_cms_extent.shp
✅ Saved vector: HEC-RAS2D_18181565_1530_cms_extent.shp
✅ Saved vector: HAND_18181565_775_cms_extent.shp
✅ Saved vector: Nencarta_18181565_775_cms_extent.shp
✅ Saved vector: Nencarta_18181565_2192_cms_extent.shp
✅ Saved vector: SRH-2D_18181565_2472_cms_extent.shp
✅ Saved vector: SRH-2D_18181565_2192_cms_extent.shp
✅ Saved vector: Nencarta_18181565_1530_cms_extent.shp
✅ Saved vector: HAND_18181565_1910_cms_extent.shp
✅ Saved vector: Nencarta_18181565_2472_cms_extent.shp
✅ Saved vector: HEC-RAS2D_18181565_1910_cms_extent.shp
✅ Saved vector: HAND_18181565_2192_cms_extent.shp
✅ Saved vector: SRH-2D_18181565_1910_cms_extent.shp
✅ Saved vector: SRH-2D_18181565_775_cms_extent.shp
✅ Saved vector: HAND_18181565_2472_cms_extent.shp
✅ Saved vector: HEC-RAS2D_18181565_2192_cms_extent.shp
✅ Saved vector: HEC-RAS2D_18181565_775_cms_extent.shp
✅

# Task 1: Preprocess the projected vector files

In [11]:
# @title Dissolve polygons, smooth boundaries, and remove all holes
import os, glob, math, shutil
import numpy as np
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon
from shapely.ops import unary_union

# ---------- Directories ----------
input_dir = '/content/extent_files'
final_dir = '/content/final_gis_files'
os.makedirs(final_dir, exist_ok=True)

# ---------- Knobs ----------
dissolve_by   = None    # e.g., "HUC8"; None = dissolve all features per file
tol_factor    = 1.5
morph_shave   = 0.0
validity_fix  = True

# ---------- Helpers ----------
def pick_metric_crs(gdf: gpd.GeoDataFrame) -> str:
    try:
        lonlat = gdf if (gdf.crs and gdf.crs.is_geographic) else gdf.to_crs(4326)
        c = lonlat.unary_union.centroid
        cx, cy = float(c.x), float(c.y)
        if -170 <= cx <= -50 and 5 <= cy <= 85:
            return "EPSG:5070"
        zone = int(math.floor((cx + 180) / 6) + 1)
        return f"EPSG:{32600 + zone}" if cy >= 0 else f"EPSG:{32700 + zone}"
    except Exception:
        return "EPSG:3857"

def median_edge_len(geom):
    lens = []
    def ring_lengths(poly: Polygon):
        xy = np.asarray(poly.exterior.coords)
        seg = np.sqrt(((xy[1:] - xy[:-1])**2).sum(axis=1))
        return seg.tolist()
    if isinstance(geom, Polygon):
        lens += ring_lengths(geom)
    elif isinstance(geom, MultiPolygon):
        for p in geom.geoms:
            lens += ring_lengths(p)
    return float(np.median(lens)) if lens else 0.0

def remove_all_holes(g):
    """Return geometry with all interior holes removed"""
    if isinstance(g, Polygon):
        return Polygon(g.exterior)
    elif isinstance(g, MultiPolygon):
        cleaned = [Polygon(p.exterior) for p in g.geoms if not p.is_empty]
        return MultiPolygon(cleaned) if len(cleaned) > 1 else (cleaned[0] if cleaned else g)
    return g

# ---------- Process all shapefiles ----------
shapefiles = glob.glob(os.path.join(input_dir, '*.shp'))
if not shapefiles:
    print(f"No .shp files found in {input_dir}")

for shp_path in shapefiles:
    try:
        base_name = os.path.splitext(os.path.basename(shp_path))[0]
        print(f"\nProcessing: {base_name}")

        gdf = gpd.read_file(shp_path)
        orig_crs = gdf.crs

        if gdf.crs is None:
            gdf.set_crs("EPSG:4326", inplace=True)

        metric_crs = pick_metric_crs(gdf)
        gdf_m = gdf.to_crs(metric_crs)

        # Dissolve
        if dissolve_by and dissolve_by in gdf_m.columns:
            dissolved = gdf_m.dissolve(by=dissolve_by, as_index=False)
        else:
            dissolved = gpd.GeoDataFrame(geometry=[unary_union(gdf_m.geometry)], crs=gdf_m.crs)

        # Simplify
        out_geoms = []
        for geom in dissolved.geometry:
            g = geom
            if morph_shave and morph_shave > 0:
                g = g.buffer(morph_shave).buffer(-morph_shave)

            tol = max(median_edge_len(g) * tol_factor, 0.0)
            g_simple = g.simplify(tol, preserve_topology=True)

            if validity_fix:
                g_simple = g_simple.buffer(0)

            # 🔹 Remove all holes
            g_simple = remove_all_holes(g_simple)

            if not g_simple.is_empty:
                out_geoms.append(g_simple)

        if not out_geoms:
            print("⚠️  No valid geometry after simplification; skipping.")
            continue

        final_metric = unary_union(out_geoms)
        out_m = gpd.GeoDataFrame(geometry=[final_metric], crs=metric_crs)

        target_crs = orig_crs if orig_crs is not None else metric_crs
        out_final = out_m.to_crs(target_crs)

        # ---------- Save shapefile ----------
        out_path = os.path.join(final_dir, f"{base_name}.shp")
        out_final.to_file(out_path)
        print(f"💾 Saved shapefile: {out_path}")

    except Exception as e:
        print(f"❌ Error processing {shp_path}: {e}")



Processing: SRH-2D_18181565_775_cms_extent


/tmp/ipykernel_23875/1052635376.py:23: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  c = lonlat.unary_union.centroid


💾 Saved shapefile: /content/final_gis_files/SRH-2D_18181565_775_cms_extent.shp

Processing: HAND_18181565_2192_cms_extent


/tmp/ipykernel_23875/1052635376.py:23: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  c = lonlat.unary_union.centroid


💾 Saved shapefile: /content/final_gis_files/HAND_18181565_2192_cms_extent.shp

Processing: SRH-2D_18181565_1530_cms_extent


...

💾 Saved shapefile: /content/final_gis_files/SRH-2D_18181565_2472_cms_extent.shp

Processing: HEC-RAS2D_18181565_775_cms_extent


/tmp/ipykernel_23875/1052635376.py:23: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  c = lonlat.unary_union.centroid


💾 Saved shapefile: /content/final_gis_files/HEC-RAS2D_18181565_775_cms_extent.shp

Processing: Nencarta_18181565_2472_cms_extent


/tmp/ipykernel_23875/1052635376.py:23: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  c = lonlat.unary_union.centroid


💾 Saved shapefile: /content/final_gis_files/Nencarta_18181565_2472_cms_extent.shp


# Task 2: Preprocess the projected raster files

In [12]:
#@title Run this code to align the rasters to same extent and compress rasters thereafter.
import os
import glob
import rasterio
from rasterio import warp
from rasterio.enums import Resampling

# Directories
input_dir = '/content/corrected_rasters'
output_dir = '/content/final_gis_files'
os.makedirs(output_dir, exist_ok=True)

# Gather all rasters directly inside input_dir (no subfolders)
raster_files = glob.glob(os.path.join(input_dir, '*.tif'))

# Use first raster as reference for alignment
with rasterio.open(raster_files[0]) as ref:
    ref_crs = ref.crs
    ref_transform = ref.transform
    ref_width = ref.width
    ref_height = ref.height
    ref_profile = ref.profile.copy()
    ref_profile.update({
        'compress': 'LZW'
    })

# Process each raster
for src_path in raster_files:
    try:
        with rasterio.open(src_path) as src:
            print(f"\nProcessing: {os.path.basename(src_path)}")

            # Ensure same CRS and align to reference grid
            if src.crs != ref_crs:
                print("❌ CRS mismatch. Skipping.")
                continue

            output_path = os.path.join(output_dir, os.path.basename(src_path))
            out_meta = ref_profile.copy()
            out_meta.update({
                'dtype': src.dtypes[0],
                'count': src.count
            })

            with rasterio.open(output_path, 'w', **out_meta) as dst:
                for i in range(1, src.count + 1):
                    warp.reproject(
                        source=rasterio.band(src, i),
                        destination=rasterio.band(dst, i),
                        src_transform=src.transform,
                        src_crs=src.crs,
                        dst_transform=ref_transform,
                        dst_crs=ref_crs,
                        resampling=Resampling.nearest
                    )
            print("✅ Aligned and compressed.")
    except Exception as e:
        print(f"❌ Error with {src_path}: {e}")



Processing: HEC-RAS2D_18181565_2472_cms_depth.tif
✅ Aligned and compressed.

Processing: HAND_18181565_1229_cms_depth.tif
✅ Aligned and compressed.

Processing: Nencarta_18181565_1910_cms_depth.tif
✅ Aligned and compressed.

Processing: HEC-RAS2D_18181565_1530_cms_depth.tif
✅ Aligned and compressed.

Processing: HAND_18181565_775_cms_depth.tif
✅ Aligned and compressed.

Processing: Nencarta_18181565_775_cms_depth.tif
✅ Aligned and compressed.

Processing: Nencarta_18181565_2192_cms_depth.tif
✅ Aligned and compressed.

Processing: SRH-2D_18181565_2472_cms_depth.tif
✅ Aligned and compressed.

Processing: SRH-2D_18181565_2192_cms_depth.tif


✅ Aligned and compressed.

Processing: Nencarta_18181565_1530_cms_depth.tif
✅ Aligned and compressed.

Processing: HAND_18181565_1910_cms_depth.tif
✅ Aligned and compressed.

Processing: Nencarta_18181565_2472_cms_depth.tif
✅ Aligned and compressed.

Processing: HEC-RAS2D_18181565_1910_cms_depth.tif
✅ Aligned and compressed.

Processing: HAND_18181565_2192_cms_depth.tif
✅ Aligned and compressed.

Processing: SRH-2D_18181565_1910_cms_depth.tif
✅ Aligned and compressed.

Processing: SRH-2D_18181565_775_cms_depth.tif
✅ Aligned and compressed.

Processing: HAND_18181565_2472_cms_depth.tif


✅ Aligned and compressed.

Processing: HEC-RAS2D_18181565_2192_cms_depth.tif
✅ Aligned and compressed.

Processing: HEC-RAS2D_18181565_775_cms_depth.tif
✅ Aligned and compressed.

Processing: HAND_18181565_1530_cms_depth.tif
✅ Aligned and compressed.

Processing: SRH-2D_18181565_1530_cms_depth.tif
✅ Aligned and compressed.

Processing: HEC-RAS2D_18181565_1229_cms_depth.tif
✅ Aligned and compressed.

Processing: SRH-2D_18181565_1229_cms_depth.tif
✅ Aligned and compressed.

Processing: Nencarta_18181565_1229_cms_depth.tif
✅ Aligned and compressed.


## Generate FIM source input file

 All input files inside the final_gis_files folder must start with one of the following valid model names:
HEC-RAS1D, HEC-RAS2D, HEC-RASCombo, SRH2D, FIER, AutoRoute, HAND, TRITON, Satellite, Surveyed, or Others.

In [14]:
import os
import pandas as pd

# Folder path
input_folder = "/content/final_gis_files"

# Approved modelname mapping from short codes to full names
modelname_mapping = {
    "HEC-RAS1D": "HEC-RAS1D",
    "HEC-RAS2D": "HEC-RAS2D",
    "HEC-RASCombo": "HEC-RAS1D/2D_Combo",
    "SRH-2D": "SRH-2D",
    "FIER": "FIER",
    "AutoRoute": "AutoRoute",
    "HAND": "HAND",
    "TRITON": "TRITON",
    "Satellite": "Satellite_Observations",
    "Surveyed": "Surveyed_Flood_Extents",
    "Nencarta": "Nencarta",
    "Others": "Others"
}

# Track valid and invalid modelnames
valid_modelnames = set()
invalid_files = []

# Step 1: Scan all files and validate model names
for filename in os.listdir(input_folder):
    if filename.endswith((".tif", ".shp")):
        parts = filename.split("_")
        if len(parts) >= 5:
            modelname = parts[0]
            if modelname in modelname_mapping:
                valid_modelnames.add(modelname)
            else:
                invalid_files.append(filename)

# Step 2: Stop and warn if there are invalid files
if invalid_files:
    print("❌ Aborted: Some filenames contain unrecognized model names:")
    for f in invalid_files:
        print(f"   ⚠️ {f}")
    print("\n🔍 Please check the file names properly. The model name must match the approved list.")
else:
    # Step 3: Proceed to generate CSV if all model names are valid
    records = []
    for model in sorted(valid_modelnames):
        records.append({
            "FIMSourceName": modelname_mapping[model],
            "CoordinateReference": "",
            "EntityName": "",
            "EntityContactEmail": "",
            "VersionNumber": "",
            "YearCreated": "",
            "EventDate": "",
            "AdditionalModelNotes": "",
            "Software": modelname_mapping[model]  # ✅ Write full name (not short code)
        })

    df = pd.DataFrame(records, columns=[
        "FIMSourceName", "CoordinateReference", "EntityName", "EntityContactEmail",
        "VersionNumber", "YearCreated", "EventDate", "AdditionalModelNotes", "Software"
    ])
    output_path = "/content/FIM_input_data.csv"
    df.to_csv(output_path, index=False)
    print(f"✅ Input file generated: {output_path}")



✅ Input file generated: /content/FIM_input_data.csv


### Create your rating curve input files for each model

In [15]:
import os
import re
import pandas as pd
from collections import defaultdict

# Directory containing all GIS files
input_folder = "/content/final_gis_files"

# Helper to parse file info
def parse_file_info(filename):
    match = re.match(r"([A-Za-z0-9\-]+)_(\d+)_(\d+)_([a-z]+)_(.+)\.(.+)", filename)
    if match:
        model, river_id, flow_str, unit, indicator, ext = match.groups()
        try:
            flow = int(flow_str)
            if unit.lower() == "cms":
                flow = round(flow * 35.3147, 0)
            return model, flow, indicator.lower(), ext.lower(), filename
        except:
            return None
    elif filename.endswith("boundary.shp"):
        parts = filename.split("_")
        model = parts[0]
        return model, None, "boundary", "shp", filename
    return None

# Organize files by model and flow
file_index = defaultdict(lambda: defaultdict(dict))  # file_index[model][flow][indicator] = filename

for fname in os.listdir(input_folder):
    parsed = parse_file_info(fname)
    if parsed:
        model, flow, indicator, ext, fname = parsed

        # Only accept .shp files for 'extent' and 'boundary'
        if indicator == "extent" and ext != "shp":
            continue
        if indicator == "boundary" and ext != "shp":
            continue

        if flow is not None:
            file_index[model][flow][indicator] = fname
        else:
            file_index[model]["boundary"]["boundary"] = fname

# Build and write rating curve CSVs per model
for model, flows in file_index.items():
    rows = []
    for flow, files in flows.items():
        if flow == "boundary":
            continue

        row = {
            "Flow": flow,
            "Depth": "",
            "ReturnPeriod": "",
            "VectorExtent": files.get("extent", ""),
            "DepthRaster": files.get("depth", ""),
            "WSERaster": files.get("wse", ""),
            "VelocityRaster": files.get("velocity", ""),
            "BoundaryVector": file_index[model].get("boundary", {}).get("boundary", "")
        }

        # Only add row if VectorExtent is .shp and DepthRaster exists
        if row["VectorExtent"].endswith(".shp") and row["DepthRaster"].endswith(".tif"):
            rows.append(row)

    if rows:
        df = pd.DataFrame(rows, columns=[
            "Flow", "Depth", "ReturnPeriod",
            "VectorExtent", "DepthRaster", "WSERaster", "VelocityRaster", "BoundaryVector"
        ])
        output_path = f"/content/{model}_ratingcurve.csv"
        df.to_csv(output_path, index=False)
        print(f"✅ Saved: {output_path}")
    else:
        print(f"⚠️ Skipped {model}: missing required files (extent.shp and depth.tif)")


✅ Saved: /content/SRH-2D_ratingcurve.csv
✅ Saved: /content/HAND_ratingcurve.csv
✅ Saved: /content/HEC-RAS2D_ratingcurve.csv
✅ Saved: /content/Nencarta_ratingcurve.csv


### Finally, download all the input files and folder

In [16]:
import zipfile
import os
from google.colab import files

# Name of the zip file
zip_path = "/content/final_outputs.zip"

# Create zip
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:

    # Add all files in final_gis_files
    for root, _, files_in_dir in os.walk("/content/final_gis_files"):
        for file in files_in_dir:
            full_path = os.path.join(root, file)
            arcname = os.path.relpath(full_path, start="/content")
            zipf.write(full_path, arcname)

    # Add all .csv files in /content
    for file in os.listdir("/content"):
        if file.endswith(".csv") and file != os.path.basename(zip_path):
            zipf.write(os.path.join("/content", file), file)

# Download
files.download(zip_path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>